In [ ]:
from pathlib import Path

import prism

from imagematerials.eol import eol_preprocess
from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import (
    EndOfLife,
    GenericMaterials,
    GenericStocks,
    Maintenance,
    MaterialIntensities,
    RestOf
)
from imagematerials.preprocessing import get_preprocessing_data
import numpy as np

from imagematerials.rest_of import rest_of_preprocessing
import matplotlib.pyplot as plt

from imagematerials.electricity.preprocessing import (
    get_preprocessing_data_gen,
    get_preprocessing_data_grid)


In [ ]:
scenario_name = "SSP2_M_CP"

In [ ]:
climate_policy_scenario_dir = Path("..", "data", "raw", "image", scenario_name)
scenario_base_path = Path("../data/raw") / 'circular_economy_scenarios'

path_image_materials = Path("C:/Coding/image-materials")

raw_data = path_image_materials / "data/raw"

In [ ]:


# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)


bld_sector = get_preprocessing_data("buildings", Path("..", "data", "raw"), 
                                    climate_policy_scenario_dir, 
                                    circular_economy_scenario_dirs = None) 
vhc_sector = get_preprocessing_data("vehicles", raw_data, 
                                    climate_policy_scenario_dir, 
                                    circular_economy_scenario_dirs = None)
rest_sector = rest_of_preprocessing(raw_data, 
                    image_scenario_directory = climate_policy_scenario_dir)

# needed for electricity model
scen = scenario_name.split("_")[0]
variant = "_".join(scenario_name.split("_")[1:])

# TODO fix this for real in the future
prep_data_vhc = vhc_sector.prep_data
prep_data_el = get_preprocessing_data_gen(path_image_materials, scen, variant)
_, prep_data_grid = get_preprocessing_data_grid(path_image_materials, scen, variant)


vhc_sector = Sector('vehicles', prep_data_vhc)
rest_sector = Sector(name='rest_of', 
                    data = rest_sector,)
sec_electr_gen = Sector("generation", prep_data_el)
sec_electr_grid = Sector("grid", prep_data_grid)

factory = ModelFactory(
[bld_sector, vhc_sector, rest_sector, sec_electr_gen, sec_electr_grid], complete_timeline
).add(GenericStocks, ["buildings", "vehicles", "generation", "grid"]
).add(GenericMaterials,  "vehicles"
).add(MaterialIntensities, ["buildings", "generation", "grid"]
).add(RestOf, "rest_of", input_sources={
    "gompertz_coefs": "rest_of",
    "gdp_per_capita": "rest_of",
    "population": "rest_of",
    "historic_diff_consumption_mean": "rest_of",
    "historic_diff_consumption_total": "rest_of"
}
)
model = factory.finish()

import warnings
with warnings.catch_warnings():
    warnings.filterwarnings("ignore")
    model.simulate(simulation_timeline)

